# 03 — Entraînement des Modèles ML

Classification état du sol + Régression humidité future.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid")
FAKE_DATA_PATH = "../../data/raw/fake_mesures.json"


## 1. Chargement & Préparation

In [ ]:
from ml.src.data.loader import load_combined
from ml.src.data.cleaner import clean
from ml.src.data.feature_builder import build_features, get_feature_columns

df = load_combined(fake_path=FAKE_DATA_PATH, prefer_api=False)
df = clean(df)
df = build_features(df)

cols = get_feature_columns()
print("Features:", cols["features"])
print(f"Dataset: {df.shape}")


## 2. Entraînement Classifieur (état du sol)

In [ ]:
from ml.src.models.train_classifier import train_classifier

clf_metrics = train_classifier(df)
print(f"\nAccuracy : {clf_metrics['accuracy']:.4f}")


## 3. Matrice de confusion

In [ ]:
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay

cm = np.array(clf_metrics["confusion_matrix"])
classes = clf_metrics["classes"]

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Matrice de Confusion — Classifieur État Sol", fontweight="bold")
plt.tight_layout()
plt.savefig("../../ml/reports/figures/06_confusion_matrix.png", bbox_inches="tight")
plt.show()


## 4. Feature Importance — Classifieur

In [ ]:
fi = pd.Series(clf_metrics["feature_importances"]).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
bars = plt.barh(fi.index, fi.values, color="#4e79a7", edgecolor="white")
plt.xlabel("Importance")
plt.title("Feature Importance — RandomForestClassifier", fontweight="bold")
plt.tight_layout()
plt.savefig("../../ml/reports/figures/07_feature_importance_clf.png", bbox_inches="tight")
plt.show()


## 5. Entraînement Régresseur (humidité future)

In [ ]:
from ml.src.models.train_regressor import train_regressor

reg_metrics = train_regressor(df)
print(f"\nMAE  : {reg_metrics['mae']:.3f}%")
print(f"RMSE : {reg_metrics['rmse']:.3f}%")
print(f"R²   : {reg_metrics['r2']:.4f}")


## 6. Résumé des performances

In [ ]:
summary = {
    "Modèle": ["RandomForestClassifier", "RandomForestRegressor"],
    "Tâche":  ["Classification état sol", "Régression humidité prévue"],
    "Métrique principale": [
        f"Accuracy : {clf_metrics['accuracy']:.4f}",
        f"MAE : {reg_metrics['mae']:.3f}%  |  R² : {reg_metrics['r2']:.4f}",
    ],
}
pd.DataFrame(summary)
